# MetS 유무별 식습관 네트워크 분석
DM/AMI/Stroke/CKD 제외 대상자 중심

In [2]:
import warnings
warnings.filterwarnings('ignore')
from network_analysis import NetworkAnalyzer
import pandas as pd
import numpy as np

# 초기화
analyzer = NetworkAnalyzer()
results_summary = {}
save_path = '../result/'
data_path = '../db/processed_data/total_only.csv'

# 데이터 로드
total_df_only = analyzer.preprocessor.load_and_prepare_data(data_path)

# MetS 비교 데이터셋 생성 (DM/AMI/Stroke/CKD 제외)
mets_datasets, filtered_data = analyzer.preprocessor.create_mets_comparison_datasets(total_df_only)

print("=== MetS 유무별 식습관 네트워크 분석 ===")
print("(DM/AMI/Stroke/CKD 제외 대상자)\n")
print(f"필터링 후 전체 대상자: {len(filtered_data):,}명")
print(f"\n- MetS(+): {len(mets_datasets['MetS_Positive_Total']):,}명 ({len(mets_datasets['MetS_Positive_Total'])/len(filtered_data)*100:.1f}%)")
print(f"  • 남성: {len(mets_datasets['MetS_Positive_Men']):,}명")
print(f"  • 여성: {len(mets_datasets['MetS_Positive_Women']):,}명")
print(f"\n- MetS(-): {len(mets_datasets['MetS_Negative_Total']):,}명 ({len(mets_datasets['MetS_Negative_Total'])/len(filtered_data)*100:.1f}%)")
print(f"  • 남성: {len(mets_datasets['MetS_Negative_Men']):,}명")
print(f"  • 여성: {len(mets_datasets['MetS_Negative_Women']):,}명")

# 네트워크 생성
mets_networks = analyzer.create_basic_networks(mets_datasets)
mets_centrality = analyzer.centrality_analyzer.analyze_network_centrality_from_graphs(mets_networks)

# 식품군 리스트
food_groups = ['Grain', 'Protein', 'Vegetables', 'Fruits', 'Dairy', 
               'Fried', 'Sweet\nFood', 'High Fat\nMeat', 'Processed\nFoods', 
               'SSB', 'Salty\nFood', 'Add-Salt']
age_groups = ['under40', '40-49', '50-64', '65plus']
age_labels = ['40세 미만', '40-49세', '50-64세', '65세 이상']

=== MetS 유무별 식습관 네트워크 분석 ===
(DM/AMI/Stroke/CKD 제외 대상자)

필터링 후 전체 대상자: 20,949명

- MetS(+): 4,637명 (22.1%)
  • 남성: 3,437명
  • 여성: 1,200명

- MetS(-): 16,312명 (77.9%)
  • 남성: 7,293명
  • 여성: 9,019명
Analyzing MetS_Positive_Total - Co-occurrence (poor)
Disconnected graph
Analyzing MetS_Positive_Total - Co-occurrence (non_poor)
Disconnected graph
Analyzing MetS_Positive_Total - Health Network (diet_disease)
Disconnected graph
Analyzing MetS_Positive_Total - Health Network (diet_mets)
Disconnected graph
Analyzing MetS_Positive_Total - Health Network (diet_biomarker)
Disconnected graph
Analyzing MetS_Negative_Total - Co-occurrence (poor)
Disconnected graph
Analyzing MetS_Negative_Total - Co-occurrence (non_poor)
Disconnected graph
Analyzing MetS_Negative_Total - Health Network (diet_disease)
Disconnected graph
Analyzing MetS_Negative_Total - Health Network (diet_mets)
Connected graph
Analyzing MetS_Negative_Total - Health Network (diet_biomarker)
Disconnected graph
Analyzing MetS_Positive_Men -

## 1. 전체 대상자: MetS 유무 비교

### 1.1 식품군 섭취 상관관계 비교

In [3]:
# 식품군 상관관계 분석
mets_corr_patterns = {}
for group_name, data in mets_datasets.items():
    if len(data) >= 100:
        patterns_df, corr_matrix = analyzer.correlation_network.analyze_diet_correlation_patterns(data)
        mets_corr_patterns[group_name] = {'patterns': patterns_df, 'matrix': corr_matrix}

print("◆ 전체 대상자: MetS 유무에 따른 식품군 섭취 상관관계 비교")
print("-" * 70)

pos_patterns = mets_corr_patterns['MetS_Positive_Total']['patterns']
neg_patterns = mets_corr_patterns['MetS_Negative_Total']['patterns']

print("\n✓ MetS(+) 그룹:")
print(f"  • 강한 정적 상관(r>0.3): {len(pos_patterns[pos_patterns['Correlation'] > 0.3])}개")
pos_strong = pos_patterns[pos_patterns['Correlation'] > 0.3].sort_values('Correlation', ascending=False)
if len(pos_strong) > 0:
    for _, row in pos_strong.head(5).iterrows():
        print(f"    - {row['Food_Group_1']} ↔ {row['Food_Group_2']}: r={row['Correlation']:.3f}")

print("\n✓ MetS(-) 그룹:")
print(f"  • 강한 정적 상관(r>0.3): {len(neg_patterns[neg_patterns['Correlation'] > 0.3])}개")
neg_strong = neg_patterns[neg_patterns['Correlation'] > 0.3].sort_values('Correlation', ascending=False)
if len(neg_strong) > 0:
    for _, row in neg_strong.head(5).iterrows():
        print(f"    - {row['Food_Group_1']} ↔ {row['Food_Group_2']}: r={row['Correlation']:.3f}")

◆ 전체 대상자: MetS 유무에 따른 식품군 섭취 상관관계 비교
----------------------------------------------------------------------

✓ MetS(+) 그룹:
  • 강한 정적 상관(r>0.3): 6개
    - Protein ↔ Vegetables: r=0.438
    - Fried ↔ High Fat
Meat: r=0.378
    - Fried ↔ Processed
Foods: r=0.373
    - Salty
Food ↔ Add-Salt: r=0.350
    - Processed
Foods ↔ SSB: r=0.343

✓ MetS(-) 그룹:
  • 강한 정적 상관(r>0.3): 5개
    - Protein ↔ Vegetables: r=0.492
    - Fried ↔ High Fat
Meat: r=0.362
    - Salty
Food ↔ Add-Salt: r=0.344
    - Processed
Foods ↔ SSB: r=0.328
    - Fried ↔ Processed
Foods: r=0.317


### 1.2 Poor/Non-Poor Diet 동시발생 패턴 비교

In [4]:
print("◆ 전체 대상자: Poor/Non-Poor Diet 동시발생 패턴")
print("-" * 70)

# 동시발생 통계 계산
mets_cooccur_stats = {}
for group_key in mets_datasets.keys():
    if group_key in mets_networks:
        mets_cooccur_stats[group_key] = {}
        for quality in ['poor', 'non_poor']:
            G = mets_networks[group_key]['cooccurrence'][quality]['graph']
            
            if G.number_of_edges() > 0:
                edge_counts = [(u, v, G.edges[u,v]['count']) for u, v in G.edges()]
                edge_counts.sort(key=lambda x: x[2], reverse=True)
            else:
                edge_counts = []
            
            mets_cooccur_stats[group_key][quality] = {
                'n_edges': G.number_of_edges(),
                'edges': edge_counts[:5],
                'total_cooccurrences': sum([e[2] for e in edge_counts])
            }

# 전체 비교
for quality, quality_name in [('poor', 'Poor Diet (1점)'), ('non_poor', 'Non-Poor Diet (3-5점)')]:
    print(f"\n✓ {quality_name}:")
    
    pos_stats = mets_cooccur_stats['MetS_Positive_Total'][quality]
    neg_stats = mets_cooccur_stats['MetS_Negative_Total'][quality]
    
    print(f"  MetS(+): 엣지 {pos_stats['n_edges']}개, 총 동시발생 {pos_stats['total_cooccurrences']:,}명")
    if pos_stats['edges']:
        print("    주요 패턴:")
        for u, v, count in pos_stats['edges'][:3]:
            print(f"      • {u} + {v}: {count:,}명")
    
    print(f"\n  MetS(-): 엣지 {neg_stats['n_edges']}개, 총 동시발생 {neg_stats['total_cooccurrences']:,}명")
    if neg_stats['edges']:
        print("    주요 패턴:")
        for u, v, count in neg_stats['edges'][:3]:
            print(f"      • {u} + {v}: {count:,}명")

◆ 전체 대상자: Poor/Non-Poor Diet 동시발생 패턴
----------------------------------------------------------------------

✓ Poor Diet (1점):
  MetS(+): 엣지 14개, 총 동시발생 6,559명
    주요 패턴:
      • Fruits + Dairy: 849명
      • Vegetables + Dairy: 848명
      • Protein + Vegetables: 791명

  MetS(-): 엣지 14개, 총 동시발생 22,310명
    주요 패턴:
      • Protein + Vegetables: 2,998명
      • Vegetables + Dairy: 2,734명
      • Fruits + Dairy: 2,562명

✓ Non-Poor Diet (3-5점):
  MetS(+): 엣지 14개, 총 동시발생 55,652명
    주요 패턴:
      • Fried + Add-Salt: 4,167명
      • Fried + High Fat
Meat: 4,146명
      • Fried + Processed
Foods: 4,112명

  MetS(-): 엣지 14개, 총 동시발생 199,019명
    주요 패턴:
      • Fried + High Fat
Meat: 14,773명
      • Fried + Add-Salt: 14,703명
      • High Fat
Meat + Add-Salt: 14,612명


### 1.3 네트워크 중심성 비교

In [5]:
print("◆ 전체 대상자: 네트워크 밀도 및 주요 허브 비교")
print("-" * 70)

for quality, quality_name in [('poor', 'Poor Diet'), ('non_poor', 'Non-Poor Diet')]:
    print(f"\n✓ {quality_name}:")
    
    pos_cooccur = mets_centrality['MetS_Positive_Total']['cooccurrence'][quality]
    neg_cooccur = mets_centrality['MetS_Negative_Total']['cooccurrence'][quality]
    
    pos_density = pos_cooccur['network_properties'].get('density', 0)
    neg_density = neg_cooccur['network_properties'].get('density', 0)
    
    print(f"  네트워크 밀도: MetS(+)={pos_density:.3f}, MetS(-)={neg_density:.3f}")
    
    pos_top3 = sorted(pos_cooccur['degree_centrality'].items(), key=lambda x: x[1], reverse=True)[:3]
    neg_top3 = sorted(neg_cooccur['degree_centrality'].items(), key=lambda x: x[1], reverse=True)[:3]
    
    print(f"  주요 허브 (MetS+): {', '.join([f'{n}({s:.2f})' for n, s in pos_top3])}")
    print(f"  주요 허브 (MetS-): {', '.join([f'{n}({s:.2f})' for n, s in neg_top3])}")

◆ 전체 대상자: 네트워크 밀도 및 주요 허브 비교
----------------------------------------------------------------------

✓ Poor Diet:
  네트워크 밀도: MetS(+)=0.212, MetS(-)=0.212
  주요 허브 (MetS+): Vegetables(0.55), Dairy(0.55), Grain(0.36)
  주요 허브 (MetS-): Dairy(0.55), Protein(0.45), Vegetables(0.45)

✓ Non-Poor Diet:
  네트워크 밀도: MetS(+)=0.212, MetS(-)=0.212
  주요 허브 (MetS+): Fried(0.45), High Fat
Meat(0.45), Processed
Foods(0.45)
  주요 허브 (MetS-): Fried(0.45), High Fat
Meat(0.45), Processed
Foods(0.45)


### 1.4 식품군별 Poor Diet 비율 비교

In [6]:
print("◆ 전체 대상자: 식품군별 Poor Diet (1점) 비율 차이")
print("-" * 70)

poor_comparison = {}
pos_data = mets_datasets['MetS_Positive_Total']
neg_data = mets_datasets['MetS_Negative_Total']

for fg in food_groups:
    if fg in pos_data.columns and fg in neg_data.columns:
        pos_poor_rate = (pos_data[fg] == 1).sum() / len(pos_data) * 100
        neg_poor_rate = (neg_data[fg] == 1).sum() / len(neg_data) * 100
        diff = pos_poor_rate - neg_poor_rate
        
        poor_comparison[fg] = {
            'MetS(+)': pos_poor_rate,
            'MetS(-)': neg_poor_rate,
            'diff': diff
        }

sorted_comparison = sorted(poor_comparison.items(), key=lambda x: abs(x[1]['diff']), reverse=True)

print("\n차이가 큰 식품군 (상위 8개):")
for fg, rates in sorted_comparison[:8]:
    direction = "↑" if rates['diff'] > 0 else "↓"
    print(f"  {fg.replace(chr(10), ' ')}: MetS(+)={rates['MetS(+)']:.1f}%, MetS(-)={rates['MetS(-)']:.1f}%, 차이={rates['diff']:+.1f}%p {direction}")

◆ 전체 대상자: 식품군별 Poor Diet (1점) 비율 차이
----------------------------------------------------------------------

차이가 큰 식품군 (상위 8개):
  Sweet Food: MetS(+)=15.1%, MetS(-)=21.1%, 차이=-6.0%p ↓
  Dairy: MetS(+)=43.5%, MetS(-)=39.7%, 차이=+3.8%p ↑
  Salty Food: MetS(+)=12.9%, MetS(-)=9.4%, 차이=+3.5%p ↑
  Fruits: MetS(+)=29.8%, MetS(-)=27.3%, 차이=+2.5%p ↑
  Grain: MetS(+)=17.2%, MetS(-)=15.4%, 차이=+1.9%p ↑
  SSB: MetS(+)=11.5%, MetS(-)=10.0%, 차이=+1.5%p ↑
  Protein: MetS(+)=23.2%, MetS(-)=24.5%, 차이=-1.2%p ↓
  High Fat Meat: MetS(+)=7.2%, MetS(-)=6.2%, 차이=+1.0%p ↑


## 2. 성별 + MetS 유무 비교

### 2.1 성별 네트워크 중심성 비교

In [7]:
print("◆ 성별 + MetS 유무: 네트워크 중심성 비교")
print("-" * 70)

for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
    print(f"\n【{gender_name}】")
    
    for quality, quality_name in [('poor', 'Poor Diet'), ('non_poor', 'Non-Poor Diet')]:
        print(f"\n  ✓ {quality_name}:")
        
        pos_key = f'MetS_Positive_{gender}'
        neg_key = f'MetS_Negative_{gender}'
        
        pos_cooccur = mets_centrality[pos_key]['cooccurrence'][quality]
        neg_cooccur = mets_centrality[neg_key]['cooccurrence'][quality]
        
        pos_density = pos_cooccur['network_properties'].get('density', 0)
        neg_density = neg_cooccur['network_properties'].get('density', 0)
        
        print(f"    밀도: MetS(+)={pos_density:.3f}, MetS(-)={neg_density:.3f}")
        
        pos_top2 = sorted(pos_cooccur['degree_centrality'].items(), key=lambda x: x[1], reverse=True)[:2]
        neg_top2 = sorted(neg_cooccur['degree_centrality'].items(), key=lambda x: x[1], reverse=True)[:2]
        
        print(f"    허브: MetS(+) {', '.join([n for n, s in pos_top2])}, MetS(-) {', '.join([n for n, s in neg_top2])}")

◆ 성별 + MetS 유무: 네트워크 중심성 비교
----------------------------------------------------------------------

【남성】

  ✓ Poor Diet:
    밀도: MetS(+)=0.212, MetS(-)=0.212
    허브: MetS(+) Fruits, Vegetables, MetS(-) Dairy, Vegetables

  ✓ Non-Poor Diet:
    밀도: MetS(+)=0.212, MetS(-)=0.212
    허브: MetS(+) Fried, Add-Salt, MetS(-) Fried, High Fat
Meat

【여성】

  ✓ Poor Diet:
    밀도: MetS(+)=0.212, MetS(-)=0.212
    허브: MetS(+) Vegetables, Dairy, MetS(-) Protein, Vegetables

  ✓ Non-Poor Diet:
    밀도: MetS(+)=0.212, MetS(-)=0.212
    허브: MetS(+) Fried, High Fat
Meat, MetS(-) Fried, High Fat
Meat


### 2.2 성별 Poor Diet 비율 비교

In [8]:
print("◆ 성별 + MetS 유무: Poor Diet 비율 차이")
print("-" * 70)

for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
    print(f"\n【{gender_name}】 차이가 큰 식품군 (상위 5개):")
    
    pos_key = f'MetS_Positive_{gender}'
    neg_key = f'MetS_Negative_{gender}'
    
    gender_comparison = {}
    for fg in food_groups:
        pos_data = mets_datasets[pos_key]
        neg_data = mets_datasets[neg_key]
        
        if fg in pos_data.columns and fg in neg_data.columns:
            pos_poor_rate = (pos_data[fg] == 1).sum() / len(pos_data) * 100
            neg_poor_rate = (neg_data[fg] == 1).sum() / len(neg_data) * 100
            diff = pos_poor_rate - neg_poor_rate
            gender_comparison[fg] = diff
    
    sorted_gender = sorted(gender_comparison.items(), key=lambda x: abs(x[1]), reverse=True)
    for fg, diff in sorted_gender[:5]:
        direction = "↑" if diff > 0 else "↓"
        print(f"  {fg.replace(chr(10), ' ')}: {diff:+.1f}%p {direction}")

◆ 성별 + MetS 유무: Poor Diet 비율 차이
----------------------------------------------------------------------

【남성】 차이가 큰 식품군 (상위 5개):
  Salty Food: +3.0%p ↑
  Sweet Food: -2.3%p ↓
  Vegetables: +2.3%p ↑
  Dairy: +1.9%p ↑
  Fruits: +1.4%p ↑

【여성】 차이가 큰 식품군 (상위 5개):
  Sweet Food: -8.0%p ↓
  Fruits: -5.7%p ↓
  Processed Foods: -2.0%p ↓
  Vegetables: -1.4%p ↓
  SSB: -1.3%p ↓


### 2.3 성별 상관관계 패턴 비교

In [9]:
print("◆ 성별 + MetS 유무: 식품군 상관관계 강도 비교")
print("-" * 70)

for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
    print(f"\n【{gender_name}】")
    
    pos_key = f'MetS_Positive_{gender}'
    neg_key = f'MetS_Negative_{gender}'
    
    if pos_key in mets_corr_patterns and neg_key in mets_corr_patterns:
        pos_g = mets_corr_patterns[pos_key]['patterns']
        neg_g = mets_corr_patterns[neg_key]['patterns']
        
        pos_strong = len(pos_g[abs(pos_g['Correlation']) > 0.3])
        neg_strong = len(neg_g[abs(neg_g['Correlation']) > 0.3])
        
        print(f"  강한 상관관계(|r|>0.3): MetS(+)={pos_strong}개, MetS(-)={neg_strong}개")

◆ 성별 + MetS 유무: 식품군 상관관계 강도 비교
----------------------------------------------------------------------

【남성】
  강한 상관관계(|r|>0.3): MetS(+)=5개, MetS(-)=6개

【여성】
  강한 상관관계(|r|>0.3): MetS(+)=4개, MetS(-)=3개


## 3. 성별 + 연령대 + MetS 유무 비교

### 3.1 연령대별 대상자 분포

In [10]:
print("◆ 성별 + 연령대 + MetS 유무: 대상자 분포")
print("-" * 70)

for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
    print(f"\n【{gender_name}】")
    
    for age, label in zip(age_groups, age_labels):
        pos_key = f'MetS_Positive_{gender}_{age}'
        neg_key = f'MetS_Negative_{gender}_{age}'
        
        pos_n = len(mets_datasets[pos_key]) if pos_key in mets_datasets else 0
        neg_n = len(mets_datasets[neg_key]) if neg_key in mets_datasets else 0
        total = pos_n + neg_n
        
        if total > 0:
            pos_pct = pos_n / total * 100
            print(f"  {label}: 총 {total:,}명 (MetS+ {pos_n:,}명 [{pos_pct:.1f}%], MetS- {neg_n:,}명 [{100-pos_pct:.1f}%])")
        else:
            print(f"  {label}: 데이터 없음")

◆ 성별 + 연령대 + MetS 유무: 대상자 분포
----------------------------------------------------------------------

【남성】
  40세 미만: 총 2,432명 (MetS+ 491명 [20.2%], MetS- 1,941명 [79.8%])
  40-49세: 총 3,415명 (MetS+ 1,120명 [32.8%], MetS- 2,295명 [67.2%])
  50-64세: 총 4,192명 (MetS+ 1,542명 [36.8%], MetS- 2,650명 [63.2%])
  65세 이상: 총 691명 (MetS+ 284명 [41.1%], MetS- 407명 [58.9%])

【여성】
  40세 미만: 총 2,580명 (MetS+ 72명 [2.8%], MetS- 2,508명 [97.2%])
  40-49세: 총 3,351명 (MetS+ 232명 [6.9%], MetS- 3,119명 [93.1%])
  50-64세: 총 3,626명 (MetS+ 640명 [17.7%], MetS- 2,986명 [82.3%])
  65세 이상: 총 662명 (MetS+ 256명 [38.7%], MetS- 406명 [61.3%])


### 3.2 연령대별 네트워크 허브 비교

In [11]:
print("◆ 성별 + 연령대 + MetS 유무: Poor Diet 주요 허브 비교")
print("-" * 70)

for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
    print(f"\n【{gender_name}】")
    
    for age, label in zip(age_groups, age_labels):
        pos_key = f'MetS_Positive_{gender}_{age}'
        neg_key = f'MetS_Negative_{gender}_{age}'
        
        print(f"\n  {label}:")
        
        # MetS(+)
        if pos_key in mets_centrality:
            pos_poor = mets_centrality[pos_key]['cooccurrence'].get('poor', {})
            if 'degree_centrality' in pos_poor and pos_poor['degree_centrality']:
                pos_top2 = sorted(pos_poor['degree_centrality'].items(), key=lambda x: x[1], reverse=True)[:2]
                pos_hubs = ', '.join([f"{n}({s:.2f})" for n, s in pos_top2])
            else:
                pos_hubs = "데이터 부족"
        else:
            pos_hubs = "분석 없음"
        
        # MetS(-)
        if neg_key in mets_centrality:
            neg_poor = mets_centrality[neg_key]['cooccurrence'].get('poor', {})
            if 'degree_centrality' in neg_poor and neg_poor['degree_centrality']:
                neg_top2 = sorted(neg_poor['degree_centrality'].items(), key=lambda x: x[1], reverse=True)[:2]
                neg_hubs = ', '.join([f"{n}({s:.2f})" for n, s in neg_top2])
            else:
                neg_hubs = "데이터 부족"
        else:
            neg_hubs = "분석 없음"
        
        print(f"    MetS(+): {pos_hubs}")
        print(f"    MetS(-): {neg_hubs}")

◆ 성별 + 연령대 + MetS 유무: Poor Diet 주요 허브 비교
----------------------------------------------------------------------

【남성】

  40세 미만:
    MetS(+): Vegetables(0.64), Fruits(0.64)
    MetS(-): Fruits(0.55), Vegetables(0.45)

  40-49세:
    MetS(+): Vegetables(0.55), Dairy(0.55)
    MetS(-): Dairy(0.64), Vegetables(0.45)

  50-64세:
    MetS(+): Dairy(0.55), Vegetables(0.45)
    MetS(-): Vegetables(0.55), Dairy(0.55)

  65세 이상:
    MetS(+): Vegetables(0.64), Dairy(0.45)
    MetS(-): Dairy(0.55), Vegetables(0.45)

【여성】

  40세 미만:
    MetS(+): Vegetables(0.64), Fruits(0.64)
    MetS(-): Sweet
Food(0.55), Vegetables(0.45)

  40-49세:
    MetS(+): Vegetables(0.64), Dairy(0.55)
    MetS(-): Protein(0.45), Vegetables(0.45)

  50-64세:
    MetS(+): Protein(0.55), Dairy(0.55)
    MetS(-): Dairy(0.55), Protein(0.45)

  65세 이상:
    MetS(+): Vegetables(0.55), Dairy(0.55)
    MetS(-): Dairy(0.55), Protein(0.45)


### 3.3 연령대별 네트워크 밀도 비교

In [12]:
print("◆ 성별 + 연령대 + MetS 유무: Poor Diet 네트워크 밀도")
print("-" * 70)

for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
    print(f"\n【{gender_name}】")
    
    for age, label in zip(age_groups, age_labels):
        pos_key = f'MetS_Positive_{gender}_{age}'
        neg_key = f'MetS_Negative_{gender}_{age}'
        
        pos_density = 0
        neg_density = 0
        
        if pos_key in mets_centrality:
            pos_density = mets_centrality[pos_key]['cooccurrence']['poor']['network_properties'].get('density', 0)
        
        if neg_key in mets_centrality:
            neg_density = mets_centrality[neg_key]['cooccurrence']['poor']['network_properties'].get('density', 0)
        
        print(f"  {label}: MetS(+)={pos_density:.3f}, MetS(-)={neg_density:.3f}")

◆ 성별 + 연령대 + MetS 유무: Poor Diet 네트워크 밀도
----------------------------------------------------------------------

【남성】
  40세 미만: MetS(+)=0.258, MetS(-)=0.212
  40-49세: MetS(+)=0.227, MetS(-)=0.212
  50-64세: MetS(+)=0.212, MetS(-)=0.212
  65세 이상: MetS(+)=0.212, MetS(-)=0.212

【여성】
  40세 미만: MetS(+)=0.273, MetS(-)=0.212
  40-49세: MetS(+)=0.242, MetS(-)=0.212
  50-64세: MetS(+)=0.212, MetS(-)=0.212
  65세 이상: MetS(+)=0.212, MetS(-)=0.212


## 4. 네트워크 시각화

In [13]:
# MetS 비교 시각화 생성 (전체, 성별, 성별+연령대)
analyzer.create_mets_comparison_plots(mets_networks, mets_centrality, save_path)

print("\n✓ MetS 비교 분석 완료")
print(f"  - 시각화 저장 위치: {save_path}MetS/")
print(f"  - 분석 그룹: {len(mets_datasets)}개")
print(f"  - 생성된 네트워크: {sum([len(mets_networks[k]) for k in mets_networks])}개")

Created Figure 1: Poor Diet Co-occurrence Networks
Created Figure 2: Non-Poor Diet Co-occurrence Networks
Created Figure 3: Men Diet Networks
Created Figure 4: Women Diet Networks
Created Figure 5a: Men Age Progression (Poor Diet)
Created Figure 5b: Women Age Progression (Poor Diet)

✓ MetS 비교 분석 완료
  - 시각화 저장 위치: ../result/MetS/
  - 분석 그룹: 22개
  - 생성된 네트워크: 44개


## 5. 결과 요약

In [14]:
print("=" * 80)
print("◆◆◆ MetS 유무별 식습관 네트워크 분석 종합 요약 ◆◆◆")
print("=" * 80)

print("\n【1. 대상자 특성】")
print(f"  • 총 분석 대상: {len(filtered_data):,}명 (DM/AMI/Stroke/CKD 제외)")
print(f"  • MetS(+): {len(mets_datasets['MetS_Positive_Total']):,}명 ({len(mets_datasets['MetS_Positive_Total'])/len(filtered_data)*100:.1f}%)")
print(f"    - 남성 {len(mets_datasets['MetS_Positive_Men']):,}명, 여성 {len(mets_datasets['MetS_Positive_Women']):,}명")
print(f"  • MetS(-): {len(mets_datasets['MetS_Negative_Total']):,}명 ({len(mets_datasets['MetS_Negative_Total'])/len(filtered_data)*100:.1f}%)")
print(f"    - 남성 {len(mets_datasets['MetS_Negative_Men']):,}명, 여성 {len(mets_datasets['MetS_Negative_Women']):,}명")

print("\n【2. 주요 발견사항】")

print("\n  ✓ 식습관 상관관계:")
pos_strong = len(pos_patterns[abs(pos_patterns['Correlation']) > 0.3])
neg_strong = len(neg_patterns[abs(neg_patterns['Correlation']) > 0.3])
print(f"    강한 상관관계(|r|>0.3): MetS(+) {pos_strong}개, MetS(-) {neg_strong}개")

print("\n  ✓ Poor Diet 네트워크 허브:")
pos_poor_hub = sorted(mets_centrality['MetS_Positive_Total']['cooccurrence']['poor']['degree_centrality'].items(), 
                      key=lambda x: x[1], reverse=True)[:3]
neg_poor_hub = sorted(mets_centrality['MetS_Negative_Total']['cooccurrence']['poor']['degree_centrality'].items(), 
                      key=lambda x: x[1], reverse=True)[:3]
print(f"    MetS(+): {', '.join([f'{n}({s:.2f})' for n, s in pos_poor_hub])}")
print(f"    MetS(-): {', '.join([f'{n}({s:.2f})' for n, s in neg_poor_hub])}")

print("\n  ✓ Non-Poor Diet 네트워크 허브:")
pos_nonpoor_hub = sorted(mets_centrality['MetS_Positive_Total']['cooccurrence']['non_poor']['degree_centrality'].items(), 
                         key=lambda x: x[1], reverse=True)[:3]
neg_nonpoor_hub = sorted(mets_centrality['MetS_Negative_Total']['cooccurrence']['non_poor']['degree_centrality'].items(), 
                         key=lambda x: x[1], reverse=True)[:3]
print(f"    MetS(+): {', '.join([f'{n}({s:.2f})' for n, s in pos_nonpoor_hub])}")
print(f"    MetS(-): {', '.join([f'{n}({s:.2f})' for n, s in neg_nonpoor_hub])}")

print("\n  ✓ Poor Diet 비율 차이가 큰 식품군 (상위 5개):")
sorted_poor_comp = sorted(poor_comparison.items(), key=lambda x: abs(x[1]['diff']), reverse=True)
for fg, rates in sorted_poor_comp[:5]:
    direction = "높음" if rates['diff'] > 0 else "낮음"
    print(f"    • {fg.replace(chr(10), ' ')}: MetS(+)가 {abs(rates['diff']):.1f}%p {direction}")

print("\n【3. 성별 특이 패턴】")
for gender, gender_name in [('Men', '남성'), ('Women', '여성')]:
    pos_key = f'MetS_Positive_{gender}'
    neg_key = f'MetS_Negative_{gender}'
    
    print(f"\n  ✓ {gender_name}:")
    
    # Poor Diet 허브
    pos_hub = sorted(mets_centrality[pos_key]['cooccurrence']['poor']['degree_centrality'].items(), 
                    key=lambda x: x[1], reverse=True)[:2]
    neg_hub = sorted(mets_centrality[neg_key]['cooccurrence']['poor']['degree_centrality'].items(), 
                    key=lambda x: x[1], reverse=True)[:2]
    print(f"    Poor Diet 허브: MetS(+) {', '.join([n for n, s in pos_hub])}, "
          f"MetS(-) {', '.join([n for n, s in neg_hub])}")
    
    # 네트워크 밀도
    pos_density = mets_centrality[pos_key]['cooccurrence']['poor']['network_properties'].get('density', 0)
    neg_density = mets_centrality[neg_key]['cooccurrence']['poor']['network_properties'].get('density', 0)
    print(f"    Poor 네트워크 밀도: MetS(+)={pos_density:.3f}, MetS(-)={neg_density:.3f}")

print("\n" + "=" * 80)

◆◆◆ MetS 유무별 식습관 네트워크 분석 종합 요약 ◆◆◆

【1. 대상자 특성】
  • 총 분석 대상: 20,949명 (DM/AMI/Stroke/CKD 제외)
  • MetS(+): 4,637명 (22.1%)
    - 남성 3,437명, 여성 1,200명
  • MetS(-): 16,312명 (77.9%)
    - 남성 7,293명, 여성 9,019명

【2. 주요 발견사항】

  ✓ 식습관 상관관계:
    강한 상관관계(|r|>0.3): MetS(+) 6개, MetS(-) 5개

  ✓ Poor Diet 네트워크 허브:
    MetS(+): Vegetables(0.55), Dairy(0.55), Grain(0.36)
    MetS(-): Dairy(0.55), Protein(0.45), Vegetables(0.45)

  ✓ Non-Poor Diet 네트워크 허브:
    MetS(+): Fried(0.45), High Fat
Meat(0.45), Processed
Foods(0.45)
    MetS(-): Fried(0.45), High Fat
Meat(0.45), Processed
Foods(0.45)

  ✓ Poor Diet 비율 차이가 큰 식품군 (상위 5개):
    • Sweet Food: MetS(+)가 6.0%p 낮음
    • Dairy: MetS(+)가 3.8%p 높음
    • Salty Food: MetS(+)가 3.5%p 높음
    • Fruits: MetS(+)가 2.5%p 높음
    • Grain: MetS(+)가 1.9%p 높음

【3. 성별 특이 패턴】

  ✓ 남성:
    Poor Diet 허브: MetS(+) Fruits, Vegetables, MetS(-) Dairy, Vegetables
    Poor 네트워크 밀도: MetS(+)=0.212, MetS(-)=0.212

  ✓ 여성:
    Poor Diet 허브: MetS(+) Vegetables, Dairy, MetS(-) Protein, Ve

## 6. 결과 저장

In [15]:
# 결과 저장
results_summary['mets_comparison'] = {
    'filtered_data_count': len(filtered_data),
    'mets_positive_count': len(mets_datasets['MetS_Positive_Total']),
    'mets_negative_count': len(mets_datasets['MetS_Negative_Total']),
    'centrality_results': mets_centrality,
    'correlation_patterns': mets_corr_patterns,
    'cooccurrence_stats': mets_cooccur_stats,
    'poor_diet_comparison': poor_comparison
}

print("✓ 결과 저장 완료")

✓ 결과 저장 완료


In [16]:
results_summary['mets_comparison']['cooccurrence_stats']

{'MetS_Positive_Total': {'poor': {'n_edges': 14,
   'edges': [('Fruits', 'Dairy', 849),
    ('Vegetables', 'Dairy', 848),
    ('Protein', 'Vegetables', 791),
    ('Vegetables', 'Fruits', 644),
    ('Protein', 'Dairy', 559)],
   'total_cooccurrences': 6559},
  'non_poor': {'n_edges': 14,
   'edges': [('Fried', 'Add-Salt', 4167),
    ('Fried', 'High Fat\nMeat', 4146),
    ('Fried', 'Processed\nFoods', 4112),
    ('High Fat\nMeat', 'Add-Salt', 4111),
    ('Processed\nFoods', 'Add-Salt', 4098)],
   'total_cooccurrences': 55652}},
 'MetS_Negative_Total': {'poor': {'n_edges': 14,
   'edges': [('Protein', 'Vegetables', 2998),
    ('Vegetables', 'Dairy', 2734),
    ('Fruits', 'Dairy', 2562),
    ('Vegetables', 'Fruits', 2188),
    ('Protein', 'Dairy', 1894)],
   'total_cooccurrences': 22310},
  'non_poor': {'n_edges': 14,
   'edges': [('Fried', 'High Fat\nMeat', 14773),
    ('Fried', 'Add-Salt', 14703),
    ('High Fat\nMeat', 'Add-Salt', 14612),
    ('Fried', 'Processed\nFoods', 14424),
    ('